# Justin Demo Notebook

This Jupyter notebook demonstrates controlling Justin, a humanoid robot, using Giskard motion control framework. The demo showcases several capabilities:

- Kinematic simulation of Justin robot using Giskard
- Picking up objects with one hand
- Handing objects between hands
- Opening/closing cabinet doors while holding objects
- Placing objects inside cabinets

The demo uses ROS (Robot Operating System) and GiskardPy for motion control. Key features of the implementation include:

- Cartesian motion goals
- Joint space motion goals
- Collision avoidance
- Multi-step manipulation sequences
- Parallel execution of motion goals
- World model updates during execution

Follow along with the step-by-step cells to see Justin perform a complete manipulation task.


Launch Giskard as a kinematic simulator.

In [1]:
%%bash --bg
roslaunch giskardpy_ros giskardpy_justin_standalone.launch

Start a node that visualized the current Motion Statechart that Giskard executed.

In [2]:
%%bash --bg
rosrun giskardpy_ros motion_statechart_inspector.py

Import necessary dependencies

In [3]:
import rospy
from geometry_msgs.msg import PoseStamped, Quaternion
import numpy as np
from giskardpy.utils.math import quaternion_from_axis_angle, quaternion_from_rotation_matrix
from giskardpy_ros.python_interface.python_interface import GiskardWrapper
from copy import deepcopy

Initialize ROS and Giskard and define default poses and joint states.

In [4]:
rospy.init_node('justin_demo')
giskard = GiskardWrapper()
default_pose = {
    "torso1_joint": 0.0,
    "torso2_joint": -0.9,
    "torso3_joint": 1.26,
    "head1_joint": 0.0,
    "head2_joint": 0.0
}
default_left_arm = {
    "left_arm1_joint": 0.41,
    "left_arm2_joint": -1.64,
    "left_arm3_joint": 0.12,
    "left_arm4_joint": 0.96,
    "left_arm5_joint": 0.71,
    "left_arm6_joint": -0.02,
    "left_arm7_joint": 0.43
}
default_right_arm = {
    "right_arm1_joint": 0.6,
    "right_arm2_joint": -1.59,
    "right_arm3_joint": 2.97,
    "right_arm4_joint": -0.99,
    "right_arm5_joint": -2.44,
    "right_arm6_joint": 0.0,
    "right_arm7_joint": 0.0,
}

better_pose = default_pose
better_pose.update(default_left_arm)
better_pose.update(default_right_arm)
right_closed = {
    "right_1thumb1_joint": 0.0,
    "right_1thumb2_joint": 0.5,
    "right_1thumb3_joint": 0.5,
    "right_3middle1_joint": 0.0,
    "right_3middle2_joint": 0.5,
    "right_3middle3_joint": 0.5,
    "right_4ring1_joint": 0.0,
    "right_4ring2_joint": 0.5,
    "right_4ring3_joint": 0.5,
    "right_2tip1_joint": 0.0,
    "right_2tip2_joint": 0.5,
    "right_2tip3_joint": 0.5,
}
left_open = {
    "left_1thumb1_joint": 0.0,
    "left_1thumb2_joint": 0.0,
    "left_1thumb3_joint": 0.0,
    "left_3middle1_joint": 0.0,
    "left_3middle2_joint": 0.0,
    "left_3middle3_joint": 0.0,
    "left_4ring1_joint": 0.0,
    "left_4ring2_joint": 0.0,
    "left_4ring3_joint": 0.0,
    "left_2tip1_joint": 0.0,
    "left_2tip2_joint": 0.0,
    "left_2tip3_joint": 0.0,
}
left_closed = {
    "left_1thumb1_joint": 0.0,
    "left_1thumb2_joint": 0.5,
    "left_1thumb3_joint": 0.5,
    "left_3middle1_joint": 0.0,
    "left_3middle2_joint": 0.5,
    "left_3middle3_joint": 0.5,
    "left_4ring1_joint": 0.0,
    "left_4ring2_joint": 0.5,
    "left_4ring3_joint": 0.5,
    "left_2tip1_joint": 0.0,
    "left_2tip2_joint": 0.5,
    "left_2tip3_joint": 0.5,
}
right_open = {
    "right_1thumb1_joint": 0.0,
    "right_1thumb2_joint": 0.0,
    "right_1thumb3_joint": 0.0,
    "right_3middle1_joint": 0.0,
    "right_3middle2_joint": 0.0,
    "right_3middle3_joint": 0.0,
    "right_4ring1_joint": 0.0,
    "right_4ring2_joint": 0.0,
    "right_4ring3_joint": 0.0,
    "right_2tip1_joint": 0.0,
    "right_2tip2_joint": 0.0,
    "right_2tip3_joint": 0.0,
}
handle_frame_id = 'dlr_kitchen/fridge_door_handle'
handle_name = 'fridge_door_handle'
door_joint = 'fridge_door_joint'
fridge = 'dlr_kitchen/fridge'
box_name = 'box'
kitchenette = 'dlr_kitchen/kitchenette'
r_tip = 'r_gripper_tool_frame'
l_tip = 'l_gripper_tool_frame'
kitchen_name = 'dlr_kitchen'

### Reset Giskard and spawn kitchen.

In [5]:
giskard.world.clear()
# spawn kitchen
kitchen_pose = PoseStamped()
kitchen_pose.header.frame_id = 'map'
kitchen_pose.pose.position.x = -2
kitchen_pose.pose.position.y = 2
kitchen_pose.pose.orientation = Quaternion(*quaternion_from_axis_angle([0, 0, 1], -np.pi / 2))
giskard.world.add_urdf(name=kitchen_name,
                       urdf=rospy.get_param('kitchen_description'),
                       pose=kitchen_pose)

# move robot to initial state
giskard.motion_goals.add_joint_position(better_pose)
giskard.add_default_end_motion_conditions()
giskard.execute()

# spawn box ontop of kitchen
box_pose = PoseStamped()
box_pose.header.frame_id = kitchenette
box_pose.pose.position.z = 0.22
box_pose.pose.position.x = -0.15
box_pose.pose.orientation.w = 1.0
giskard.world.add_box(name=box_name, size=(0.03, 0.15, 0.2), pose=box_pose, parent_link=kitchenette)
giskard.world.dye_group(group_name=box_name, rgba=(0.0, 0.0, 1.0, 1.0))

error_codes: 0

### Step 1: Grasp Object

In [6]:
pre_grasp_pose = PoseStamped()
pre_grasp_pose.header.frame_id = box_name
pre_grasp_pose.pose.orientation = Quaternion(0, 1, 0, 0)
pre_grasp_pose.pose.position.z = 0.35
box_pre_grasped = 'pregrasp pose'
giskard.motion_goals.add_cartesian_pose(name=box_pre_grasped,
                                        goal_pose=pre_grasp_pose,
                                        tip_link='r_gripper_tool_frame',
                                        root_link='map', end_condition=box_pre_grasped)

grasp_pose = deepcopy(pre_grasp_pose)
grasp_pose.pose.position.z -= 0.2
box_grasped = giskard.motion_goals.add_cartesian_pose(name='grasp box',
                                                      goal_pose=grasp_pose,
                                                      tip_link='r_gripper_tool_frame',
                                                      root_link='map',
                                                      start_condition=box_pre_grasped)
right_hand_closed = giskard.motion_goals.add_joint_position(name='close right hand',
                                                            goal_state=right_closed,
                                                            start_condition=box_pre_grasped)

giskard.monitors.add_end_motion(start_condition=f'{box_grasped} and {right_hand_closed}')
# dlr_kitchen_setup.motion_goals.allow_self_collision(end_condition=box_pre_grasped)
giskard.motion_goals.allow_collision(group1='rollin_justin', group2=box_name)
giskard.motion_goals.allow_self_collision()
giskard.motion_goals.add_justin_torso_limit(name='torso4_joint', end_condition='')
result = giskard.execute()

error: 
  type: ''
  msg: ''
trajectory: 
  header: 
    seq: 0
    stamp: 
      secs: 0
      nsecs:         0
    frame_id: ''
  joint_names: 
    - torso1_joint
    - torso2_joint
    - torso3_joint
    - head1_joint
    - head2_joint
    - left_arm1_joint
    - left_arm2_joint
    - left_arm3_joint
    - left_arm4_joint
    - left_arm5_joint
    - left_arm6_joint
    - left_arm7_joint
    - left_1thumb1_joint
    - left_1thumb2_joint
    - left_1thumb3_joint
    - left_1thumb3_joint
    - left_2tip1_joint
    - left_2tip2_joint
    - left_2tip3_joint
    - left_2tip3_joint
    - left_3middle1_joint
    - left_3middle2_joint
    - left_3middle3_joint
    - left_3middle3_joint
    - left_4ring1_joint
    - left_4ring2_joint
    - left_4ring3_joint
    - left_4ring3_joint
    - right_arm1_joint
    - right_arm2_joint
    - right_arm3_joint
    - right_arm4_joint
    - right_arm5_joint
    - right_arm6_joint
    - right_arm7_joint
    - right_1thumb1_joint
    - right_1thumb2_joint
  

### Step 1.1: attach grasped object
The world model cannot change mid motion, so we have to have a break here

In [7]:
giskard.world.update_parent_link_of_group(name=box_name, parent_link='r_gripper_tool_frame')

error: 
  type: ''
  msg: ''

### Step 2: Hand box over to left hand.

In [8]:
base_pose = PoseStamped()
base_pose.header.frame_id = 'base_footprint'
base_pose.pose.orientation.w = 1.0
base_pose.pose.position.x = -1
drove_back = giskard.motion_goals.add_cartesian_pose(name='drive back',
                                                     goal_pose=base_pose,
                                                     tip_link='base_footprint',
                                                     root_link='map')

hand_over_pose = PoseStamped()
hand_over_pose.header.frame_id = 'l_gripper_tool_frame'
hand_over_pose.pose.orientation = Quaternion(1, 0, 0, 0)
hand_over_pose.pose.position.z = 0.3
handed_over = 'hand over'
giskard.motion_goals.add_cartesian_pose(name=handed_over,
                                        goal_pose=hand_over_pose,
                                        tip_link='r_gripper_tool_frame',
                                        root_link='l_gripper_tool_frame',
                                        end_condition=handed_over)
left_hand_opened = 'open left hand'
giskard.motion_goals.add_joint_position(name=left_hand_opened,
                                        goal_state=left_open,
                                        end_condition=left_hand_opened)
left_hand_closed = 'close left hand'
giskard.motion_goals.add_joint_position(name=left_hand_closed,
                                        goal_state=left_closed,
                                        start_condition=f'{handed_over} and {left_hand_opened}',
                                        end_condition=left_hand_closed)
right_hand_opened = 'open right hand'
giskard.motion_goals.add_joint_position(name=right_hand_opened,
                                        goal_state=right_open,
                                        start_condition=f'{left_hand_closed}',
                                        end_condition=right_hand_opened)

giskard.monitors.add_end_motion(start_condition=f'{right_hand_opened} and {drove_back}')
giskard.motion_goals.allow_collision(group1='rollin_justin', group2=box_name)
giskard.motion_goals.allow_self_collision()
giskard.motion_goals.add_justin_torso_limit(name='torso4_joint', end_condition='')
result = giskard.execute()

error: 
  type: ''
  msg: ''
trajectory: 
  header: 
    seq: 0
    stamp: 
      secs: 0
      nsecs:         0
    frame_id: ''
  joint_names: 
    - torso1_joint
    - torso2_joint
    - torso3_joint
    - head1_joint
    - head2_joint
    - left_arm1_joint
    - left_arm2_joint
    - left_arm3_joint
    - left_arm4_joint
    - left_arm5_joint
    - left_arm6_joint
    - left_arm7_joint
    - left_1thumb1_joint
    - left_1thumb2_joint
    - left_1thumb3_joint
    - left_1thumb3_joint
    - left_2tip1_joint
    - left_2tip2_joint
    - left_2tip3_joint
    - left_2tip3_joint
    - left_3middle1_joint
    - left_3middle2_joint
    - left_3middle3_joint
    - left_3middle3_joint
    - left_4ring1_joint
    - left_4ring2_joint
    - left_4ring3_joint
    - left_4ring3_joint
    - right_arm1_joint
    - right_arm2_joint
    - right_arm3_joint
    - right_arm4_joint
    - right_arm5_joint
    - right_arm6_joint
    - right_arm7_joint
    - right_1thumb1_joint
    - right_1thumb2_joint
  

### Step 2.1: Tell Giskard the box is now attached to the left hand.

In [9]:
giskard.world.update_parent_link_of_group(name=box_name, parent_link='l_gripper_tool_frame')

error: 
  type: ''
  msg: ''

### Step 3: Open the fridge and start placing the object one the fridge is half open.

In [10]:
giskard.cancel_all_goals()
giskard.clear_motion_goals_and_monitors()
in_default_pose = 'default joint pose'
giskard.motion_goals.add_joint_position(name=in_default_pose,
                                        goal_state=default_pose,
                                        end_condition=in_default_pose)
handle_grasp_pose = PoseStamped()
handle_grasp_pose.header.frame_id = handle_frame_id
handle_grasp_pose.pose.orientation = Quaternion(*quaternion_from_rotation_matrix([[0, 0, 1, 0],
                                                                                  [1, 0, 0, 0],
                                                                                  [0, 1, 0, 0],
                                                                                  [0, 0, 0, 1]]))
handle_grasp_pose.pose.position.x = -0.12
handle_graped = 'grasp handle'
giskard.motion_goals.add_cartesian_pose(name=handle_graped,
                                        goal_pose=handle_grasp_pose,
                                        tip_link=r_tip,
                                        root_link='map',
                                        start_condition=in_default_pose,
                                        end_condition=handle_graped)
right_hand_closed = 'close right hand'
giskard.motion_goals.add_joint_position(name=right_hand_closed,
                                        goal_state=right_closed,
                                        start_condition=f'{handle_graped}',
                                        end_condition=right_hand_closed)

door_open = 'open fridge'
giskard.motion_goals.add_open_container(name=door_open,
                                        tip_link=r_tip,
                                        environment_link=handle_name,
                                        start_condition=right_hand_closed,
                                        end_condition='')
door_half_open = 'is door half open?'
giskard.monitors.add_joint_position_above(name=door_half_open,
                                          joint_name=door_joint,
                                          threshold=np.pi / 4,
                                          start_condition=handle_graped,
                                          end_condition=door_half_open)

place_pose = PoseStamped()
place_pose.header.frame_id = fridge
place_pose.pose.orientation = Quaternion(*quaternion_from_rotation_matrix([[0, 0, 1, 0],
                                                                           [1, 0, 0, 0],
                                                                           [0, 1, 0, 0],
                                                                           [0, 0, 0, 1]]))
place_pose.pose.position.z = 0.4
place_pose.pose.position.x = 0.
box_placed = 'place box'
giskard.motion_goals.add_cartesian_pose(name=box_placed,
                                        goal_pose=place_pose,
                                        tip_link=box_name,
                                        root_link='map',
                                        start_condition=door_half_open,
                                        end_condition=box_placed)
left_hand_opened = 'open left hand'
giskard.motion_goals.add_joint_position(name=left_hand_opened,
                                        goal_state=left_open,
                                        start_condition=f'{box_placed}',
                                        end_condition=left_hand_opened)

done = f'{door_open} and {left_hand_opened}'
giskard.monitors.add_end_motion(start_condition=done)
giskard.motion_goals.allow_collision(group2='right_hand',
                                     group1='dlr_kitchen')
giskard.motion_goals.allow_self_collision()
giskard.motion_goals.add_justin_torso_limit(name='torso4_joint', end_condition='')
result = giskard.execute()

error: 
  type: ''
  msg: ''
trajectory: 
  header: 
    seq: 0
    stamp: 
      secs: 0
      nsecs:         0
    frame_id: ''
  joint_names: 
    - torso1_joint
    - torso2_joint
    - torso3_joint
    - head1_joint
    - head2_joint
    - left_arm1_joint
    - left_arm2_joint
    - left_arm3_joint
    - left_arm4_joint
    - left_arm5_joint
    - left_arm6_joint
    - left_arm7_joint
    - left_1thumb1_joint
    - left_1thumb2_joint
    - left_1thumb3_joint
    - left_1thumb3_joint
    - left_2tip1_joint
    - left_2tip2_joint
    - left_2tip3_joint
    - left_2tip3_joint
    - left_3middle1_joint
    - left_3middle2_joint
    - left_3middle3_joint
    - left_3middle3_joint
    - left_4ring1_joint
    - left_4ring2_joint
    - left_4ring3_joint
    - left_4ring3_joint
    - right_arm1_joint
    - right_arm2_joint
    - right_arm3_joint
    - right_arm4_joint
    - right_arm5_joint
    - right_arm6_joint
    - right_arm7_joint
    - right_1thumb1_joint
    - right_1thumb2_joint
  

### Step 3.1: Tell Giskard the box is now attached to the fridge instead of the robot.

In [11]:
giskard.world.update_parent_link_of_group(name=box_name, parent_link=fridge)

error: 
  type: ''
  msg: ''

### Step 4: Retract the left hand and close the fridge.

In [12]:
retract_pose = PoseStamped()
retract_pose.header.frame_id = 'l_gripper_tool_frame'
retract_pose.pose.orientation.w = 1.0
retract_pose.pose.position.z = -0.5
retracted = 'retract left hand'
giskard.motion_goals.add_joint_position(name=retracted,
                                        goal_state=default_left_arm,
                                        end_condition=retracted)
delay = giskard.monitors.add_sleep(name='wait 1s', seconds=1)
giskard.motion_goals.add_open_container(name='hold handle',
                                        tip_link='r_gripper_tool_frame',
                                        environment_link=handle_name,
                                        end_condition=delay)

door_closed = giskard.motion_goals.add_close_container(name='close fridge',
                                                       tip_link='r_gripper_tool_frame',
                                                       environment_link=handle_name,
                                                       start_condition=delay)
giskard.monitors.add_end_motion(start_condition=f'{door_closed} and {retracted}')
giskard.motion_goals.allow_collision(group2='right_hand',
                                     group1='dlr_kitchen')
giskard.motion_goals.allow_self_collision()
giskard.motion_goals.add_justin_torso_limit(name='torso4_joint', end_condition='')
result = giskard.execute()

error: 
  type: ''
  msg: ''
trajectory: 
  header: 
    seq: 0
    stamp: 
      secs: 0
      nsecs:         0
    frame_id: ''
  joint_names: 
    - torso1_joint
    - torso2_joint
    - torso3_joint
    - head1_joint
    - head2_joint
    - left_arm1_joint
    - left_arm2_joint
    - left_arm3_joint
    - left_arm4_joint
    - left_arm5_joint
    - left_arm6_joint
    - left_arm7_joint
    - left_1thumb1_joint
    - left_1thumb2_joint
    - left_1thumb3_joint
    - left_1thumb3_joint
    - left_2tip1_joint
    - left_2tip2_joint
    - left_2tip3_joint
    - left_2tip3_joint
    - left_3middle1_joint
    - left_3middle2_joint
    - left_3middle3_joint
    - left_3middle3_joint
    - left_4ring1_joint
    - left_4ring2_joint
    - left_4ring3_joint
    - left_4ring3_joint
    - right_arm1_joint
    - right_arm2_joint
    - right_arm3_joint
    - right_arm4_joint
    - right_arm5_joint
    - right_arm6_joint
    - right_arm7_joint
    - right_1thumb1_joint
    - right_1thumb2_joint
  